In [1]:
import pandas as pd
import numpy as np

In [2]:
equity_df = pd.read_csv('equity_value_data_v2.csv')
equity_df

,timestamp,close_equity,user_id,date,daily_equity_change
0,2016-08-18T00:00:00Z,1211.6055,0012db34aa7b083f5714e7831195e54d,2016-08-18,NaN
1,2016-08-19T00:00:00Z,1173.5640,0012db34aa7b083f5714e7831195e54d,2016-08-19,-38.0415
2,2016-08-22T00:00:00Z,1253.0597,0012db34aa7b083f5714e7831195e54d,2016-08-22,79.4957
3,2016-08-23T00:00:00Z,1252.9050,0012db34aa7b083f5714e7831195e54d,2016-08-23,-0.1547
4,2016-08-24T00:00:00Z,1262.1360,0012db34aa7b083f5714e7831195e54d,2016-08-24,9.2310
...,...,...,...,...,...
1119153,2017-08-14T00:00:00Z,2156.2400,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-14,47.9100
1119154,2017-08-15T00:00:00Z,2134.7100,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-15,-21.5300
1119155,2017-08-16T00:00:00Z,2152.1200,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-16,17.4100
1119156,2017-08-17T00:00:00Z,2042.2800,ffc1e622f3a0b2666f09a6dcb7f27918,2017-08-17,-109.8400


In [ ]:
# Function to create full date range for one user
def complete_user_dates(user_df):
    full_dates = pd.DataFrame({'date': pd.date_range(user_df['date'].min(), user_df['date'].max())})
    full_dates['user_id'] = user_df['user_id'].iloc[0]
    return full_dates

complete_dates_df = (
    equity_df.groupby('user_id')
    .apply(complete_user_dates)
    .reset_index(drop=True)
)

equity_df['date'] = pd.to_datetime(equity_df['date'])
complete_dates_df['date'] = pd.to_datetime(complete_dates_df['date'])

In [4]:
# Merge to bring in actual trades
full_equity_df = complete_dates_df.merge(
    equity_df[['user_id', 'date', 'close_equity', 'timestamp']],
    on=['user_id', 'date'],
    how='left'
)

# Sort new DataFrame
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Add binary flag for active/inactive days
full_equity_df['active_day'] = full_equity_df['timestamp'].notna().astype(int)

# HOW TO TREAT CLOSE_EQUITY ON INACTIVE DAYS?
# Fill null close_equity with $9
full_equity_df['close_equity'] = full_equity_df['close_equity'].fillna(9)

# DAILY CHANGE? DOES THIS MAKE SENSE IF I AM LEAVING INACTIVE DAYS NULL?
# Daily change
# full_equity_df['daily_equity_change'] = full_equity_df.groupby('user_id')['close_equity'].diff()

In [5]:
# Drop timestamp as it no longer has useful information after date was split off.
full_equity_df.drop('timestamp', axis=1, inplace=True)

In [6]:
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Create a group that increments every time we hit an active day
grp = (
    full_equity_df['active_day']
    .eq(1)
    .groupby(full_equity_df['user_id'])
    .cumsum()
)

# Count days since last active day
full_equity_df['days_since_last_active'] = (
    full_equity_df
    .groupby(['user_id', grp])
    .cumcount()
)

In [7]:
full_equity_df['streak_at_least_28'] = (full_equity_df['days_since_last_active'] >= 28).astype(int)

In [9]:
# Sort by user and date first
full_equity_df = full_equity_df.sort_values(['user_id', 'date'])

# Use transform instead of apply
full_equity_df['ever_above_10_prev'] = full_equity_df.groupby('user_id')['close_equity'] \
                                                .transform(lambda x: x.shift(1).ge(10).cummax())

# Fill NaN for the first row per user
full_equity_df['ever_above_10_prev'] = full_equity_df['ever_above_10_prev'].fillna(False)

full_equity_df['ever_above_10_prev'] = full_equity_df['ever_above_10_prev'].astype(int)

full_equity_df

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,ever_above_10_prev
0,2016-08-18,0012db34aa7b083f5714e7831195e54d,1211.6055,1,0,0,0
1,2016-08-19,0012db34aa7b083f5714e7831195e54d,1173.5640,1,0,0,1
2,2016-08-20,0012db34aa7b083f5714e7831195e54d,9.0000,0,1,0,1
3,2016-08-21,0012db34aa7b083f5714e7831195e54d,9.0000,0,2,0,1
4,2016-08-22,0012db34aa7b083f5714e7831195e54d,1253.0597,1,0,0,1
...,...,...,...,...,...,...,...
1647990,2017-08-14,ffc1e622f3a0b2666f09a6dcb7f27918,2156.2400,1,0,0,1
1647991,2017-08-15,ffc1e622f3a0b2666f09a6dcb7f27918,2134.7100,1,0,0,1
1647992,2017-08-16,ffc1e622f3a0b2666f09a6dcb7f27918,2152.1200,1,0,0,1
1647993,2017-08-17,ffc1e622f3a0b2666f09a6dcb7f27918,2042.2800,1,0,0,1


# Churned users DataFrame

In [16]:
churned_users_df = full_equity_df[full_equity_df['streak_at_least_28'] == 1].copy()
churned_users_df = churned_users_df[churned_users_df['ever_above_10_prev'] == 1]
churned_users_df.drop_duplicates(subset=['user_id'], keep='first', inplace=True)
churned_users_df

churned_users = 

,date,user_id,close_equity,active_day,days_since_last_active,streak_at_least_28,ever_above_10_prev
1543,2017-03-27,00440034cc4152bfb01b30f5c381c4e3,9.0,0,28,1,1
2031,2016-12-02,005d630a68b4ab3a2f4cd49d9a87c50d,9.0,0,28,1,1
13247,2016-12-12,028367ff3cbcc04c2afc2ce3336c00e2,9.0,0,28,1,1
26172,2016-11-23,0423b88554cedaa7efd8dd4c81774cce,9.0,0,28,1,1
37276,2016-10-13,062ea0ff3b7fc36ae471968aced1f4a1,9.0,0,28,1,1
...,...,...,...,...,...,...,...
1635288,2017-06-09,fdc54af66d1190dec81b95b4a2965634,9.0,0,28,1,1
1642126,2017-05-01,ff0ae95285c43e3a5af84860bffaa544,9.0,0,28,1,1
1642992,2016-10-27,ff377467d4e28b425266a8b2c8b2f5c7,9.0,0,28,1,1
1645248,2017-02-09,ff7610fdd7ac5cbfa0b17aca53af5db4,9.0,0,28,1,1


# Features


In [18]:
features_df = pd.read_csv('features_data.csv')
features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91
...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a


In [19]:
features_df['churned'] = features_df['user_id'].isin(churned_users_df['user_id']).astype(int)
features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,churned
0,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,33.129417,stock,40.0,med_time_horizon,895044c23edc821881e87da749c01034,0
1,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,16.573517,stock,200.0,short_time_horizon,458b1d95441ced242949deefe8e4b638,0
2,med_risk_tolerance,limited_investment_exp,very_important_liq_need,iOS,10.008367,stock,25.0,long_time_horizon,c7936f653d293479e034865db9bb932f,0
3,med_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,1.031633,stock,100.0,short_time_horizon,b255d4bd6c9ba194d3a350b3e76c6393,0
4,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.187250,stock,20.0,long_time_horizon,4a168225e89375b8de605cbc0977ae91,0
...,...,...,...,...,...,...,...,...,...,...
5579,high_risk_tolerance,limited_investment_exp,very_important_liq_need,Android,8.339283,stock,300.0,long_time_horizon,03880c726d8a4e5db006afe4119ad974,0
5580,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,7.241383,stock,100.0,short_time_horizon,ae8315109657f44852b24c6bca4decd6,1
5581,med_risk_tolerance,no_investment_exp,very_important_liq_need,both,22.967167,stock,50.0,short_time_horizon,f29c174989f9737058fe808fcf264135,0
5582,med_risk_tolerance,limited_investment_exp,somewhat_important_liq_need,iOS,10.338417,stock,100.0,long_time_horizon,24843497d1de88b2e7233f694436cb3a,0


In [21]:
features_df.dtypes

risk_tolerance                   object
investment_experience            object
liquidity_needs                  object
platform                         object
time_spent                      float64
instrument_type_first_traded     object
first_deposit_amount            float64
time_horizon                     object
user_id                          object
churned                           int64
dtype: object

In [26]:
features_df['time_horizon'].value_counts()

time_horizon
short_time_horizon    2833
long_time_horizon     1833
med_time_horizon       918
Name: count, dtype: int64

In [28]:
# Cleaning up values so that dummy column names will be cleaner.

# risk tolerance
tolerance_map = {
    'high_risk_tolerance': 'high',
    'med_risk_tolerance': 'medium',
    'low_risk_tolerance': 'low'
}
features_df['risk_tolerance'] = features_df['risk_tolerance'].replace(tolerance_map)

# experience
experience_map = {
    'limited_investment_exp': 'limited',
    'no_investment_exp': 'none',
    'good_investment_exp': 'good',
    'extensive_investment_exp': 'extensive'
}
features_df['investment_experience'] = features_df['investment_experience'].replace(experience_map)

# liquidity
liquidity_map = {
    'very_important_liq_need': 'very_important',
    'somewhat_important_liq_need': 'somewhat_important',
    'not_important_liq_need': 'not_important'
}
features_df['liquidity_needs'] = features_df['liquidity_needs'].replace(liquidity_map)

# time horizon
horizon_map = {
    'short_time_horizon': 'short',
    'long_time_horizon': 'long',
    'med_time_horizon': 'medium'
}
features_df['time_horizon'] = features_df['time_horizon'].replace(horizon_map)

In [39]:
# Reduce uncommon instruments traded into one 'other' category
values_to_replace = ['cef', 'wrt', '0', 'rlt', 'lp', 'tracking']
replacement_value = 'other'
features_df['instrument_type_first_traded'] = features_df['instrument_type_first_traded'].replace(values_to_replace, replacement_value)

features_df

,risk_tolerance,investment_experience,liquidity_needs,platform,time_spent,instrument_type_first_traded,first_deposit_amount,time_horizon,user_id,churned
0,high,limited,very_important,Android,33.129417,stock,40.0,medium,895044c23edc821881e87da749c01034,0
1,medium,limited,very_important,Android,16.573517,stock,200.0,short,458b1d95441ced242949deefe8e4b638,0
2,medium,limited,very_important,iOS,10.008367,stock,25.0,long,c7936f653d293479e034865db9bb932f,0
3,medium,limited,very_important,Android,1.031633,stock,100.0,short,b255d4bd6c9ba194d3a350b3e76c6393,0
4,high,limited,very_important,Android,8.187250,stock,20.0,long,4a168225e89375b8de605cbc0977ae91,0
...,...,...,...,...,...,...,...,...,...,...
5579,high,limited,very_important,Android,8.339283,stock,300.0,long,03880c726d8a4e5db006afe4119ad974,0
5580,medium,limited,somewhat_important,iOS,7.241383,stock,100.0,short,ae8315109657f44852b24c6bca4decd6,1
5581,medium,none,very_important,both,22.967167,stock,50.0,short,f29c174989f9737058fe808fcf264135,0
5582,medium,limited,somewhat_important,iOS,10.338417,stock,100.0,long,24843497d1de88b2e7233f694436cb3a,0


In [32]:
# Get Dummies, except for user_id which will be ignored in modeling
non_numeric_cols = features_df.select_dtypes(exclude=[np.number, bool]).columns.tolist()
col_to_exclude = 'user_id'
cols_for_dummies = [col for col in non_numeric_cols if col != col_to_exclude]

features_with_dummies = pd.get_dummies(features_df, columns=cols_for_dummies, dtype=int)

In [35]:
features_with_dummies.columns

Index(['time_spent', 'first_deposit_amount', 'user_id', 'churned',
       'risk_tolerance_high', 'risk_tolerance_low', 'risk_tolerance_medium',
       'investment_experience_extensive', 'investment_experience_good',
       'investment_experience_limited', 'investment_experience_none',
       'liquidity_needs_not_important', 'liquidity_needs_somewhat_important',
       'liquidity_needs_very_important', 'platform_Android', 'platform_both',
       'platform_iOS', 'instrument_type_first_traded_0',
       'instrument_type_first_traded_adr', 'instrument_type_first_traded_cef',
       'instrument_type_first_traded_etp', 'instrument_type_first_traded_lp',
       'instrument_type_first_traded_mlp', 'instrument_type_first_traded_reit',
       'instrument_type_first_traded_rlt',
       'instrument_type_first_traded_stock',
       'instrument_type_first_traded_tracking',
       'instrument_type_first_traded_wrt', 'time_horizon_long',
       'time_horizon_medium', 'time_horizon_short'],
      dtype

In [36]:
features_df['instrument_type_first_traded'].value_counts()

instrument_type_first_traded
stock       4827
etp          383
adr          197
mlp           55
reit          55
cef           20
wrt           16
0             13
rlt            9
lp             8
tracking       1
Name: count, dtype: int64

# Did any users churn multiple times? is that worth noting?